<a href="https://colab.research.google.com/github/rathorebharat/mftracker/blob/main/mf-correction-engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import requests
import os

In [14]:
# @title
import requests
import pandas as pd
import numpy as np

# ============================================================
# CONFIGURATION
# ============================================================

codes = [
    119063, 141877, 152430, 122639, 147409,
    120594, 120578, 119705, 127042, 138528,
    119218, 119727, 119151, 148623, 120733,
    118834, 135781, 120334, 118825, 153198,
    125354, 120244, 119544, 118989, 130503,
    119714
]

API_URL = "https://api.tigzig.com/mf/v1/nav"


# ============================================================
# GET ALL HISTORICAL NAV DATA
# ============================================================

response = requests.get(
    API_URL,
    params={
        "schemes": ",".join(map(str, codes))
    },
    timeout=120
)

response.raise_for_status()

result = response.json()

print("Requested:", len(codes))
print("Returned :", result["count"])
print("Not found:", result["not_found"])


# ============================================================
# CONVERT API RESPONSE TO DATAFRAME
# ============================================================

records = []

for scheme_data in result["schemes"]:

    scheme_code = scheme_data["scheme_code"]
    scheme_name = scheme_data["scheme_name"]

    for row in scheme_data["data"]:

        records.append({
            "Scheme_Code": scheme_code,
            "Scheme_Name": scheme_name,
            "Date": row["date"],
            "NAV": row["nav"]
        })

df_nav = pd.DataFrame(records)

df_nav["Date"] = pd.to_datetime(df_nav["Date"])

df_nav["NAV"] = pd.to_numeric(
    df_nav["NAV"],
    errors="coerce"
)

df_nav = (
    df_nav
    .dropna(
        subset=[
            "Scheme_Code",
            "Date",
            "NAV"
        ]
    )
    .sort_values(
        [
            "Scheme_Code",
            "Date"
        ]
    )
    .drop_duplicates(
        [
            "Scheme_Code",
            "Date"
        ]
    )
    .reset_index(drop=True)
)

print(df_nav.shape)
print(df_nav.head())

print("Schemes requested :", len(codes))
print("Schemes returned  :", len(result["schemes"]))
print("Not found         :", result["not_found"])

print("\nReturned scheme codes:")

print([
    x["scheme_code"]
    for x in result["schemes"]
])



Requested: 26
Returned : 26
Not found: []
(72369, 4)
   Scheme_Code                                   Scheme_Name       Date     NAV
0       118825  Mirae Asset Large Cap Fund - Direct Plan ... 2013-01-02  18.972
1       118825  Mirae Asset Large Cap Fund - Direct Plan ... 2013-01-03  19.011
2       118825  Mirae Asset Large Cap Fund - Direct Plan ... 2013-01-04  19.008
3       118825  Mirae Asset Large Cap Fund - Direct Plan ... 2013-01-07  18.950
4       118825  Mirae Asset Large Cap Fund - Direct Plan ... 2013-01-08  18.954
Schemes requested : 26
Schemes returned  : 26
Not found         : []

Returned scheme codes:
[118825, 118834, 118989, 119063, 119151, 119218, 119544, 119705, 119714, 119727, 120244, 120334, 120578, 120594, 120733, 122639, 125354, 127042, 130503, 135781, 138528, 141877, 147409, 148623, 152430, 153198]


In [20]:
# @title
# ================================================================
# V8.3 — ₹2 LAKH INVESTMENT ALLOCATION ENGINE
# ================================================================

MONTHLY_INVESTMENT_INR = 200000


# ------------------------------------------------
# 1. INVESTMENT SCORE
# ------------------------------------------------
#
# Stronger correction = higher score
# But group overlap reduces the score when several
# funds represent essentially the same exposure.
# ------------------------------------------------

def investment_score(row):

    signal = row.get("Signal", "NORMAL")

    correction = safe_float(
        row.get("Current_Correction_%", np.nan)
    )

    p75 = safe_float(
        row.get("75th_Correction_%", np.nan)
    )

    p90 = safe_float(
        row.get("90th_Correction_%", np.nan)
    )

    p95 = safe_float(
        row.get("95th_Correction_%", np.nan)
    )

    group_pressure = safe_float(
        row.get("Net_Group_Pressure", 0)
    )

    score = 0.0

    # ------------------------------------------------------------
    # SIGNAL
    # ------------------------------------------------------------

    signal_score = {
        "STRONG ACCUMULATE": 100,
        "ACCUMULATE": 70,
        "WATCH": 30,
        "NORMAL": 0,
        "TRIM WATCH": -50,
        "TRIM": -80,
        "STRONG TRIM": -100
    }

    score += signal_score.get(signal, 0)


    # ------------------------------------------------------------
    # CORRECTION DEPTH
    # ------------------------------------------------------------

    if not pd.isna(correction):

        if not pd.isna(p95) and correction >= p95:
            score += 60

        elif not pd.isna(p90) and correction >= p90:
            score += 45

        elif not pd.isna(p75) and correction >= p75:
            score += 30

        elif not pd.isna(p75) and correction >= p75 * 0.75:
            score += 15


    # ------------------------------------------------------------
    # GROUP PRESSURE
    # ------------------------------------------------------------

    if not pd.isna(group_pressure):

        # Positive group pressure supports buying
        score += group_pressure * 10


    return max(score, 0)


# ------------------------------------------------
# 2. CALCULATE RAW SCORES
# ------------------------------------------------

diagnostics_df["Investment_Score"] = (
    diagnostics_df.apply(
        investment_score,
        axis=1
    )
)


# ------------------------------------------------
# 3. ONLY BUY-ELIGIBLE FUNDS
# ------------------------------------------------

buy_mask = diagnostics_df["Signal"].isin(
    [
        "STRONG ACCUMULATE",
        "ACCUMULATE",
        "WATCH"
    ]
)

buy_df = diagnostics_df[
    buy_mask
].copy()


# ------------------------------------------------
# 4. REMOVE ZERO-SCORE FUNDS
# ------------------------------------------------

buy_df = buy_df[
    buy_df["Investment_Score"] > 0
].copy()


# ------------------------------------------------
# 5. GROUP OVERLAP CONTROL
# ------------------------------------------------
#
# Multiple funds inside the same strategic group
# should NOT automatically receive equal allocations.
#
# Example:
#
# HDFC Nifty 50
# ICICI Nifty 50
#
# are effectively one exposure.
#
# Therefore the group gets a budget first.
# ------------------------------------------------

if len(buy_df) > 0:

    group_scores = (
        buy_df
        .groupby("Strategic_Group")[
            "Investment_Score"
        ]
        .sum()
    )

else:

    group_scores = pd.Series(dtype=float)


# ------------------------------------------------
# 6. GROUP ALLOCATION
# ------------------------------------------------

if len(group_scores) > 0:

    total_group_score = group_scores.sum()

    group_budget = (
        group_scores
        / total_group_score
        * MONTHLY_INVESTMENT_INR
    )

else:

    group_budget = pd.Series(
        dtype=float
    )


# ------------------------------------------------
# 7. FUND ALLOCATION WITHIN GROUP
# ------------------------------------------------

buy_df["Group_Budget_INR"] = (
    buy_df["Strategic_Group"]
    .map(group_budget)
    .fillna(0)
)


buy_df["Within_Group_Weight"] = (
    buy_df["Investment_Score"]
    /
    buy_df.groupby(
        "Strategic_Group"
    )["Investment_Score"]
    .transform("sum")
)


buy_df["Investment_INR"] = (
    buy_df["Group_Budget_INR"]
    *
    buy_df["Within_Group_Weight"]
)


# ------------------------------------------------
# 8. ROUND TO ₹1,000
# ------------------------------------------------

buy_df["Investment_INR"] = (
    buy_df["Investment_INR"]
    .div(1000)
    .round()
    .mul(1000)
)


# ------------------------------------------------
# 9. CAP INDIVIDUAL FUND ALLOCATION
# ------------------------------------------------
#
# No single fund gets > 25% of monthly capital.
# This prevents one extreme correction from consuming
# the entire ₹2 lakh.
# ------------------------------------------------

MAX_FUND_ALLOCATION_INR = (
    MONTHLY_INVESTMENT_INR * 0.25
)


buy_df["Investment_INR"] = (
    buy_df["Investment_INR"]
    .clip(
        upper=MAX_FUND_ALLOCATION_INR
    )
)


# ------------------------------------------------
# 10. FINAL NORMALIZATION
# ------------------------------------------------

allocated = buy_df[
    "Investment_INR"
].sum()


if allocated > 0:

    scale = (
        MONTHLY_INVESTMENT_INR
        / allocated
    )

    buy_df["Investment_INR"] = (
        buy_df["Investment_INR"]
        * scale
    )


    buy_df["Investment_INR"] = (
        buy_df["Investment_INR"]
        .div(1000)
        .round()
        .mul(1000)
    )


# ------------------------------------------------
# 11. ADD ₹2 LAKH ALLOCATION BACK TO FULL OUTPUT
# ------------------------------------------------

diagnostics_df["Investment_INR"] = 0


if len(buy_df) > 0:

    diagnostics_df.loc[
        buy_df.index,
        "Investment_INR"
    ] = buy_df["Investment_INR"]


# ------------------------------------------------
# 12. INVESTMENT PERCENTAGE
# ------------------------------------------------

diagnostics_df["Investment_%"] = (
    diagnostics_df["Investment_INR"]
    / MONTHLY_INVESTMENT_INR
    * 100
)


# ------------------------------------------------
# 13. FINAL INVESTMENT ACTION
# ------------------------------------------------

def investment_action(row):

    amount = row["Investment_INR"]

    signal = row["Signal"]

    if amount <= 0:
        return "NO INVESTMENT"

    if signal == "STRONG ACCUMULATE":
        return "INVEST AGGRESSIVELY"

    if signal == "ACCUMULATE":
        return "INVEST"

    if signal == "WATCH":
        return "SMALL INVESTMENT"

    return "NO INVESTMENT"


diagnostics_df[
    "Investment_Action"
] = diagnostics_df.apply(
    investment_action,
    axis=1
)


# ================================================================
# 14. FINAL PORTFOLIO OUTPUT
# ================================================================

portfolio_columns = [

    "Scheme_Code",
    "Scheme_Name",
    "Strategic_Group",

    "Signal",

    "Current_NAV",
    "Current_Correction_%",
    "Correction_Zone",
    "Cycle_Phase",
    "Peak_Recovery_%",

    "Trough_Growth_%",
    "Overshoot_%",

    "Profit_Taking_Signal",
    "Suggested_Trim_%",

    "Group_Signal",
    "Buy_Pressure_%",
    "Trim_Pressure_%",
    "Net_Group_Pressure",

    "Investment_Score",
    "Investment_Action",
    "Investment_INR",
    "Investment_%",

    "Rationale"
]


portfolio_output = diagnostics_df[
    [
        c for c in portfolio_columns
        if c in diagnostics_df.columns
    ]
].copy()


# ================================================================
# 15. DISPLAY ONLY FUNDS RECEIVING MONEY
# ================================================================

investment_output = (
    portfolio_output[
        portfolio_output["Investment_INR"] > 0
    ]
    .sort_values(
        "Investment_INR",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\n"
    + "=" * 150
)

print(
    "V8.3 — ₹2 LAKH MONTHLY INVESTMENT ALLOCATION"
)

print(
    "=" * 150
)

display(
    investment_output
)


print(
    "\n"
    + "=" * 150
)

print(
    f"TOTAL ALLOCATED: "
    f"₹{investment_output['Investment_INR'].sum():,.0f}"
)

print(
    f"UNALLOCATED: "
    f"₹{MONTHLY_INVESTMENT_INR - investment_output['Investment_INR'].sum():,.0f}"
)

print(
    "=" * 150
)


V8.3 — ₹2 LAKH MONTHLY INVESTMENT ALLOCATION


,Scheme_Code,Scheme_Name,Strategic_Group,Signal,Current_NAV,Current_Correction_%,Correction_Zone,Cycle_Phase,Peak_Recovery_%,Trough_Growth_%,Overshoot_%,Profit_Taking_Signal,Suggested_Trim_%,Group_Signal,Buy_Pressure_%,Trim_Pressure_%,Net_Group_Pressure,Investment_Score,Investment_Action,Investment_INR,Investment_%,Rationale
0,152430,HDFC NIFTY200 Momentum 30 Index Fund - Di...,FACTOR_EQUITY,STRONG ACCUMULATE,10.5573,17.510783,ABOVE P95,EXTREME CORRECTION,9.883791,2.383746,-17.510783,NO TRIM,0,GROUP STRONG ACCUMULATE,100.0,0.0,3.00,190.0,INVEST AGGRESSIVELY,17000,8.5,Correction 17.51% | Median 1.39% | P75 2....
1,138528,PGIM India Global Equity Opportunities Fu...,INTERNATIONAL,STRONG ACCUMULATE,56.0600,11.199113,ABOVE P95,EXTREME CORRECTION,2.884615,0.376007,-11.199113,NO TRIM,0,GROUP STRONG ACCUMULATE,100.0,0.0,3.00,190.0,INVEST AGGRESSIVELY,17000,8.5,Correction 11.20% | Median 1.05% | P75 2....
2,120594,ICICI Prudential Technology Fund - Direct...,SECTOR_EQUITY,STRONG ACCUMULATE,205.3000,17.876715,ABOVE P95,EXTREME CORRECTION,-1.753188,-0.373659,-17.876715,NO TRIM,0,GROUP ACCUMULATE,71.4,0.0,1.86,178.6,INVEST AGGRESSIVELY,16000,8.0,Correction 17.88% | Median 0.93% | P75 2....
3,120578,SBI TECHNOLOGY OPPORTUNITIES FUND - DIREC...,SECTOR_EQUITY,STRONG ACCUMULATE,235.1978,12.550009,ABOVE P95,EXTREME CORRECTION,-4.314314,-0.590041,-12.550009,NO TRIM,0,GROUP ACCUMULATE,71.4,0.0,1.86,178.6,INVEST AGGRESSIVELY,16000,8.0,Correction 12.55% | Median 0.97% | P75 2....
4,119063,HDFC Nifty 50 Index Fund - Direct Plan,CORE_EQUITY,STRONG ACCUMULATE,235.7810,7.827910,ABOVE P95,EXTREME CORRECTION,-6.727997,-0.532519,-7.827910,NO TRIM,0,GROUP ACCUMULATE,75.0,0.0,1.75,177.5,INVEST AGGRESSIVELY,16000,8.0,Correction 7.83% | Median 0.91% | P75 1.8...
5,120244,ICICI Prudential Banking and Financial Se...,SECTOR_EQUITY,STRONG ACCUMULATE,149.6900,5.944078,ABOVE P95,EXTREME CORRECTION,10.922787,0.780987,-5.944078,NO TRIM,0,GROUP ACCUMULATE,71.4,0.0,1.86,178.6,INVEST AGGRESSIVELY,16000,8.0,Correction 5.94% | Median 1.01% | P75 2.2...
6,127042,Motilal Oswal Midcap Fund-Direct Plan-Gro...,GROWTH_EQUITY,STRONG ACCUMULATE,122.7222,6.077745,ABOVE P95,EXTREME CORRECTION,1.491019,0.098041,-6.077745,NO TRIM,0,GROUP ACCUMULATE,60.0,0.0,1.60,176.0,INVEST AGGRESSIVELY,15000,7.5,Correction 6.08% | Median 0.96% | P75 2.0...
7,122639,Parag Parikh Flexi Cap Fund - Direct Plan...,GROWTH_EQUITY,STRONG ACCUMULATE,91.0500,4.931651,ABOVE P95,EXTREME CORRECTION,12.051244,0.715907,-4.931651,NO TRIM,0,GROUP ACCUMULATE,60.0,0.0,1.60,176.0,INVEST AGGRESSIVELY,15000,7.5,Correction 4.93% | Median 0.62% | P75 1.3...
8,118825,Mirae Asset Large Cap Fund - Direct Plan ...,CORE_EQUITY,ACCUMULATE,128.5680,4.220987,P90 → P95,EXTREME CORRECTION,-10.837246,-0.429052,-4.220987,NO TRIM,0,GROUP ACCUMULATE,75.0,0.0,1.75,132.5,INVEST,12000,6.0,Correction 4.22% | Median 0.86% | P75 1.7...
9,119218,DSP Large & Mid Cap Fund - Direct Plan - ...,CORE_EQUITY,ACCUMULATE,705.6970,3.389659,P75 → P90,DEEP CORRECTION,-1.964337,-0.067547,-3.389659,NO TRIM,0,GROUP ACCUMULATE,75.0,0.0,1.75,117.5,INVEST,10000,5.0,Correction 3.39% | Median 0.89% | P75 1.9...



TOTAL ALLOCATED: ₹200,000
UNALLOCATED: ₹0


In [17]:
# @title
# =================================================================================================
# V8.4 — EPISODE-BASED BUYING OPPORTUNITY ROBUSTNESS ENGINE
#
# Built on:
#   - existing df_nav
#   - existing V8.x / V7 correction signal logic
#
# IMPORTANT:
#   This is a DIAGNOSTIC / ROBUSTNESS layer.
#   It does NOT replace the existing correction/profit/group engines.
#
# Main correction from previous version:
#   Historical opportunities are represented by CORRECTION EPISODES,
#   NOT by individual daily NAV observations.
#
# Episode:
#       PEAK
#         ↓
#       correction
#         ↓
#       TROUGH
#         ↓
#       recovery
#
# Each episode is counted ONCE.
# =================================================================================================

import pandas as pd
import numpy as np
from datetime import timedelta


# -------------------------------------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------------------------------------

ROBUSTNESS_NEAR_TOLERANCE = 1.00       # percentage points
MIN_CORRECTION_FOR_EPISODE = 3.0       # ignore tiny noise
RECOVERY_THRESHOLD = 0.995             # recover to 99.5% of peak
MIN_EPISODE_GAP_DAYS = 5

# Confidence based on number of COMPLETED historical episodes
def episode_confidence(n):
    if n < 10:
        return "LOW"
    elif n < 20:
        return "MEDIUM"
    else:
        return "HIGH"


# -------------------------------------------------------------------------------------------------
# SIGNAL CLASSIFIER
#
# This deliberately uses the same conceptual thresholds as the correction engine:
#
#   below P50        NORMAL
#   P50-P75          WATCH
#   P75-P90          ACCUMULATE
#   P90-P95          STRONG ACCUMULATE
#   > P95            STRONG ACCUMULATE
#
# NOTE:
# Your existing V8.x signal engine can remain the authoritative signal engine.
# This classifier is only used to reconstruct the historical progression.
# -------------------------------------------------------------------------------------------------

def historical_signal(correction, p50, p75, p90, p95):

    if pd.isna(correction):
        return "UNKNOWN"

    if correction < p50:
        return "NORMAL"

    elif correction < p75:
        return "WATCH"

    elif correction < p90:
        return "ACCUMULATE"

    elif correction < p95:
        return "STRONG ACCUMULATE"

    else:
        return "STRONG ACCUMULATE"


# -------------------------------------------------------------------------------------------------
# PREPARE NAV DATA
# -------------------------------------------------------------------------------------------------

def prepare_nav(df_nav):

    df = df_nav.copy()

    # Normalize column names
    df.columns = [str(c).strip() for c in df.columns]

    # Required columns
    required = {"Scheme_Code", "Date", "NAV"}

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"df_nav is missing required columns: {sorted(missing)}"
        )

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["NAV"] = pd.to_numeric(df["NAV"], errors="coerce")

    df = df.dropna(subset=["Scheme_Code", "Date", "NAV"])

    df = df[df["NAV"] > 0].copy()

    df["Scheme_Code"] = pd.to_numeric(
        df["Scheme_Code"], errors="coerce"
    ).astype(int)

    df = (
        df
        .sort_values(["Scheme_Code", "Date"])
        .drop_duplicates(
            subset=["Scheme_Code", "Date"],
            keep="last"
        )
        .reset_index(drop=True)
    )

    return df


# -------------------------------------------------------------------------------------------------
# BUILD EPISODES FOR ONE FUND
#
# Important design:
#
# We first identify local peaks and troughs using a state machine.
#
# We do NOT generate one "opportunity" for every daily NAV.
#
# A correction episode begins when NAV falls sufficiently from a peak.
# The episode ends when NAV recovers sufficiently toward that peak.
#
# If another higher peak is established before the correction becomes
# meaningful, the peak is simply updated.
# -------------------------------------------------------------------------------------------------

def build_correction_episodes(
    fund_df,
    min_correction=MIN_CORRECTION_FOR_EPISODE,
    recovery_threshold=RECOVERY_THRESHOLD
):

    df = fund_df.sort_values("Date").reset_index(drop=True).copy()

    if len(df) < 3:
        return pd.DataFrame()

    nav = df["NAV"].astype(float).values
    dates = df["Date"].values

    episodes = []

    peak_idx = 0
    trough_idx = 0

    in_correction = False

    i = 1

    while i < len(df):

        current_nav = nav[i]

        # -----------------------------------------------------------------------------------------
        # STATE 1: NOT CURRENTLY IN A CORRECTION
        # -----------------------------------------------------------------------------------------

        if not in_correction:

            # New high
            if current_nav >= nav[peak_idx]:
                peak_idx = i
                trough_idx = i
                i += 1
                continue

            correction = (
                (nav[peak_idx] - current_nav)
                / nav[peak_idx]
                * 100
            )

            # Meaningful correction begins
            if correction >= min_correction:

                in_correction = True
                trough_idx = i

            i += 1
            continue

        # -----------------------------------------------------------------------------------------
        # STATE 2: CORRECTION IS OPEN
        # -----------------------------------------------------------------------------------------

        # New lower trough
        if current_nav < nav[trough_idx]:
            trough_idx = i
            i += 1
            continue

        peak_nav = nav[peak_idx]

        # Recovery relative to original peak
        recovery_ratio = current_nav / peak_nav

        # -----------------------------------------------------------------------------------------
        # CASE A:
        # Full recovery
        # -----------------------------------------------------------------------------------------

        if recovery_ratio >= recovery_threshold:

            trough_nav = nav[trough_idx]

            correction_pct = (
                (peak_nav - trough_nav)
                / peak_nav
                * 100
            )

            if correction_pct >= min_correction:

                peak_date = pd.Timestamp(dates[peak_idx])
                trough_date = pd.Timestamp(dates[trough_idx])
                recovery_date = pd.Timestamp(dates[i])

                episodes.append({

                    "Peak_Date": peak_date,
                    "Peak_NAV": peak_nav,

                    "Trough_Date": trough_date,
                    "Trough_NAV": trough_nav,

                    "Recovery_Date": recovery_date,
                    "Recovery_NAV": current_nav,

                    "Correction_%": correction_pct,

                    "Peak_to_Trough_Days":
                        (trough_date - peak_date).days,

                    "Trough_to_Recovery_Days":
                        (recovery_date - trough_date).days,

                    "Total_Recovery_Days":
                        (recovery_date - peak_date).days,

                    "Recovered":
                        (
                            (current_nav - trough_nav)
                            / (peak_nav - trough_nav)
                            * 100
                        )
                    })

            # Recovery creates a new reference peak
            peak_idx = i
            trough_idx = i
            in_correction = False

            i += 1
            continue

        # -----------------------------------------------------------------------------------------
        # CASE B:
        # NAV rises but has not yet recovered.
        #
        # Keep the same correction episode.
        # -----------------------------------------------------------------------------------------

        i += 1

    # ---------------------------------------------------------------------------------------------
    # OPEN EPISODE
    # ---------------------------------------------------------------------------------------------

    if in_correction:

        trough_nav = nav[trough_idx]
        peak_nav = nav[peak_idx]

        correction_pct = (
            (peak_nav - trough_nav)
            / peak_nav
            * 100
        )

        if correction_pct >= min_correction:

            current_nav = nav[-1]

            recovered_pct = (
                (current_nav - trough_nav)
                / (peak_nav - trough_nav)
                * 100
            )

            episodes.append({

                "Peak_Date":
                    pd.Timestamp(dates[peak_idx]),

                "Peak_NAV":
                    peak_nav,

                "Trough_Date":
                    pd.Timestamp(dates[trough_idx]),

                "Trough_NAV":
                    trough_nav,

                "Recovery_Date":
                    pd.NaT,

                "Recovery_NAV":
                    np.nan,

                "Correction_%":
                    correction_pct,

                "Peak_to_Trough_Days":
                    (
                        pd.Timestamp(dates[trough_idx])
                        - pd.Timestamp(dates[peak_idx])
                    ).days,

                "Trough_to_Recovery_Days":
                    np.nan,

                "Total_Recovery_Days":
                    np.nan,

                "Recovered":
                    recovered_pct
            })

    return pd.DataFrame(episodes)


# -------------------------------------------------------------------------------------------------
# HISTORICAL PERCENTILES
#
# CRITICAL:
# For an episode ending at date T, thresholds are calculated only from
# episodes completed BEFORE T.
#
# This prevents look-ahead bias.
# -------------------------------------------------------------------------------------------------

def calculate_historical_thresholds(
    completed_episodes
):

    if completed_episodes is None or len(completed_episodes) == 0:

        return {
            "P50": np.nan,
            "P75": np.nan,
            "P90": np.nan,
            "P95": np.nan,
            "N": 0
        }

    corrections = (
        completed_episodes["Correction_%"]
        .dropna()
        .astype(float)
    )

    if len(corrections) == 0:

        return {
            "P50": np.nan,
            "P75": np.nan,
            "P90": np.nan,
            "P95": np.nan,
            "N": 0
        }

    return {
        "P50": corrections.quantile(0.50),
        "P75": corrections.quantile(0.75),
        "P90": corrections.quantile(0.90),
        "P95": corrections.quantile(0.95),
        "N": len(corrections)
    }


# -------------------------------------------------------------------------------------------------
# ADD SIGNAL PROGRESSION TO EACH EPISODE
#
# Instead of asking:
#
#   "What signal was shown on every historical day?"
#
# we ask:
#
#   "What was the strongest signal reached during this correction episode?"
#
# This is the correct unit for robustness analysis.
# -------------------------------------------------------------------------------------------------

def enrich_episode_signals(
    episodes
):

    if episodes is None or len(episodes) == 0:
        return episodes

    episodes = episodes.copy()

    episodes["P50"] = np.nan
    episodes["P75"] = np.nan
    episodes["P90"] = np.nan
    episodes["P95"] = np.nan
    episodes["Historical_Episodes_Available"] = 0

    episodes["Signal_at_50"] = ""
    episodes["Signal_at_75"] = ""
    episodes["Signal_at_90"] = ""
    episodes["Signal_at_95"] = ""

    episodes["Peak_to_Trough_Signal"] = ""

    for i in range(len(episodes)):

        episode_date = episodes.loc[i, "Trough_Date"]

        previous = episodes.iloc[:i]

        thresholds = calculate_historical_thresholds(previous)

        p50 = thresholds["P50"]
        p75 = thresholds["P75"]
        p90 = thresholds["P90"]
        p95 = thresholds["P95"]

        episodes.loc[i, "P50"] = p50
        episodes.loc[i, "P75"] = p75
        episodes.loc[i, "P90"] = p90
        episodes.loc[i, "P95"] = p95
        episodes.loc[i, "Historical_Episodes_Available"] = thresholds["N"]

        correction = episodes.loc[i, "Correction_%"]

        if thresholds["N"] >= 3:

            episodes.loc[i, "Peak_to_Trough_Signal"] = historical_signal(
                correction,
                p50,
                p75,
                p90,
                p95
            )

        else:

            episodes.loc[i, "Peak_to_Trough_Signal"] = "INSUFFICIENT_HISTORY"

    return episodes


# -------------------------------------------------------------------------------------------------
# FIND HISTORICAL EPISODES COMPARABLE TO TODAY
#
# Two classifications:
#
#   DEEPER_THAN_TODAY
#       historical maximum correction >= today's correction
#
#   NEAR_TODAY
#       historical maximum correction >= today's correction - tolerance
#
# This avoids mixing the two concepts.
# -------------------------------------------------------------------------------------------------

def historical_opportunities_for_fund(
    fund_df,
    current_nav=None,
    current_date=None
):

    fund_df = fund_df.sort_values("Date").reset_index(drop=True)

    if len(fund_df) == 0:
        return {}, pd.DataFrame()

    if current_date is None:
        current_date = fund_df["Date"].max()

    current_date = pd.Timestamp(current_date)

    if current_nav is None:
        current_nav = float(
            fund_df.loc[
                fund_df["Date"] == current_date,
                "NAV"
            ].iloc[-1]
        )

    # ---------------------------------------------------------------------------------------------
    # CURRENT PEAK
    # ---------------------------------------------------------------------------------------------

    before_current = fund_df[
        fund_df["Date"] <= current_date
    ].copy()

    peak_idx = before_current["NAV"].idxmax()

    current_peak = float(
        before_current.loc[peak_idx, "NAV"]
    )

    current_peak_date = pd.Timestamp(
        before_current.loc[peak_idx, "Date"]
    )

    current_correction = (
        (current_peak - current_nav)
        / current_peak
        * 100
    )

    # ---------------------------------------------------------------------------------------------
    # BUILD HISTORICAL EPISODES
    # ---------------------------------------------------------------------------------------------

    episodes = build_correction_episodes(
        fund_df[
            fund_df["Date"] <= current_date
        ]
    )

    if len(episodes) == 0:

        return {
            "Current_NAV": current_nav,
            "Current_Peak": current_peak,
            "Current_Peak_Date": current_peak_date,
            "Current_Correction_%": current_correction,
            "Historical_Episode_Count": 0,
            "Confidence": "LOW"
        }, pd.DataFrame()

    # Do NOT use an open episode as historical evidence.
    completed = episodes[
        episodes["Recovery_Date"].notna()
    ].copy()

    open_episode = episodes[
        episodes["Recovery_Date"].isna()
    ].copy()

    # ---------------------------------------------------------------------------------------------
    # Only completed episodes are used for historical comparison.
    # ---------------------------------------------------------------------------------------------

    completed = enrich_episode_signals(completed)

    # Remove any episode that begins too close to current date if needed
    completed = completed.sort_values(
        "Trough_Date"
    ).reset_index(drop=True)

    # ---------------------------------------------------------------------------------------------
    # CURRENT HISTORICAL THRESHOLDS
    # ---------------------------------------------------------------------------------------------

    thresholds = calculate_historical_thresholds(
        completed
    )

    p50 = thresholds["P50"]
    p75 = thresholds["P75"]
    p90 = thresholds["P90"]
    p95 = thresholds["P95"]

    current_signal = historical_signal(
        current_correction,
        p50,
        p75,
        p90,
        p95
    ) if thresholds["N"] >= 3 else "INSUFFICIENT_HISTORY"

    # ---------------------------------------------------------------------------------------------
    # COMPARISON
    # ---------------------------------------------------------------------------------------------

    opportunities = completed.copy()

    opportunities["Comparison"] = np.where(
        opportunities["Correction_%"] >= current_correction,
        "DEEPER_THAN_TODAY",
        np.where(
            opportunities["Correction_%"]
            >= current_correction - ROBUSTNESS_NEAR_TOLERANCE,
            "NEAR_TODAY",
            "SHALLOWER"
        )
    )

    # Only useful historical opportunities
    opportunities = opportunities[
        opportunities["Comparison"].isin(
            ["DEEPER_THAN_TODAY", "NEAR_TODAY"]
        )
    ].copy()

    # ---------------------------------------------------------------------------------------------
    # SORT:
    # deepest first
    # ---------------------------------------------------------------------------------------------

    opportunities = opportunities.sort_values(
        ["Correction_%", "Trough_Date"],
        ascending=[False, True]
    ).reset_index(drop=True)

    # ---------------------------------------------------------------------------------------------
    # CURRENT DIAGNOSTIC
    # ---------------------------------------------------------------------------------------------

    diagnostic = {

        "Current_Date":
            current_date,

        "Current_NAV":
            current_nav,

        "Current_Peak":
            current_peak,

        "Current_Peak_Date":
            current_peak_date,

        "Current_Correction_%":
            current_correction,

        "P50":
            p50,

        "P75":
            p75,

        "P90":
            p90,

        "P95":
            p95,

        "Current_Signal":
            current_signal,

        "Historical_Episode_Count":
            thresholds["N"],

        "Confidence":
            episode_confidence(thresholds["N"]),

        "Deeper_Than_Today_Count":
            int(
                (
                    completed["Correction_%"]
                    >= current_correction
                ).sum()
            ),

        "Near_Today_Count":
            int(
                (
                    (
                        completed["Correction_%"]
                        < current_correction
                    )
                    &
                    (
                        completed["Correction_%"]
                        >= current_correction
                        - ROBUSTNESS_NEAR_TOLERANCE
                    )
                ).sum()
            ),

        "Open_Historical_Episode":
            len(open_episode) > 0
    }

    return diagnostic, opportunities


# -------------------------------------------------------------------------------------------------
# RUN FOR ONE FUND
# -------------------------------------------------------------------------------------------------

def robustness_test(
    df_nav,
    scheme_code
):

    df = prepare_nav(df_nav)

    fund = df[
        df["Scheme_Code"] == int(scheme_code)
    ].copy()

    if len(fund) == 0:
        raise ValueError(
            f"Scheme code {scheme_code} not found in df_nav"
        )

    scheme_name = (
        fund["Scheme_Name"].iloc[0]
        if "Scheme_Name" in fund.columns
        else "NoName"
    )

    diagnostic, opportunities = historical_opportunities_for_fund(
        fund
    )

    diagnostic["Scheme_Code"] = int(scheme_code)
    diagnostic["Scheme_Name"] = scheme_name

    # Put identifiers first
    ordered_diag = {
        "Scheme_Code": diagnostic.pop("Scheme_Code"),
        "Scheme_Name": diagnostic.pop("Scheme_Name"),
        **diagnostic
    }

    return ordered_diag, opportunities


# -------------------------------------------------------------------------------------------------
# PRINT REPORT
# -------------------------------------------------------------------------------------------------

def print_robustness_report(
    df_nav,
    scheme_code
):

    diagnostic, opportunities = robustness_test(
        df_nav,
        scheme_code
    )

    print()
    print("=" * 110)
    print(
        f"BUYING OPPORTUNITY ROBUSTNESS — EPISODE ENGINE"
    )
    print("=" * 110)

    print(f"Fund              : {diagnostic['Scheme_Name']}")
    print(f"Scheme Code       : {diagnostic['Scheme_Code']}")
    print(f"Current Date      : {diagnostic['Current_Date'].date()}")
    print(f"Current NAV       : {diagnostic['Current_NAV']:.4f}")
    print(
        f"Current Peak      : {diagnostic['Current_Peak']:.4f}"
    )
    print(
        f"Current Peak Date : {diagnostic['Current_Peak_Date'].date()}"
    )

    print(
        f"Current Correction: "
        f"{diagnostic['Current_Correction_%']:.2f}%"
    )

    print()
    print("HISTORICAL THRESHOLDS")
    print("-" * 110)

    for k in ["P50", "P75", "P90", "P95"]:

        value = diagnostic[k]

        if pd.isna(value):
            print(f"{k:<8}: N/A")
        else:
            print(f"{k:<8}: {value:.2f}%")

    print()
    print("CURRENT SIGNAL")
    print("-" * 110)

    print(
        f"{diagnostic['Current_Signal']}"
    )

    print()
    print("HISTORICAL SAMPLE")
    print("-" * 110)

    print(
        f"Completed episodes : "
        f"{diagnostic['Historical_Episode_Count']}"
    )

    print(
        f"Confidence          : "
        f"{diagnostic['Confidence']}"
    )

    print(
        f"Deeper than today   : "
        f"{diagnostic['Deeper_Than_Today_Count']}"
    )

    print(
        f"Near today          : "
        f"{diagnostic['Near_Today_Count']}"
    )

    print()
    print("=" * 110)
    print("HISTORICAL EPISODES COMPARABLE TO TODAY")
    print("=" * 110)

    if opportunities.empty:

        print(
            "No completed historical correction episode "
            "was as deep as or near today's correction."
        )

        return diagnostic, opportunities

    cols = [
        "Peak_Date",
        "Peak_NAV",
        "Trough_Date",
        "Trough_NAV",
        "Recovery_Date",
        "Correction_%",
        "Peak_to_Trough_Days",
        "Trough_to_Recovery_Days",
        "Total_Recovery_Days",
        "Comparison",
        "Peak_to_Trough_Signal",
        "Historical_Episodes_Available"
    ]

    available_cols = [
        c for c in cols
        if c in opportunities.columns
    ]

    display_df = opportunities[
        available_cols
    ].copy()

    pd.set_option(
        "display.max_rows",
        100
    )

    pd.set_option(
        "display.width",
        220
    )

    print(
        display_df.to_string(
            index=False
        )
    )

    print()
    print("=" * 110)
    print("EPISODE INTERPRETATION")
    print("=" * 110)

    deeper = opportunities[
        opportunities["Comparison"]
        == "DEEPER_THAN_TODAY"
    ]

    near = opportunities[
        opportunities["Comparison"]
        == "NEAR_TODAY"
    ]

    print(
        f"Today's correction: "
        f"{diagnostic['Current_Correction_%']:.2f}%"
    )

    print(
        f"Historical episodes deeper than today: "
        f"{len(deeper)}"
    )

    print(
        f"Historical episodes within "
        f"{ROBUSTNESS_NEAR_TOLERANCE:.1f} percentage point: "
        f"{len(near)}"
    )

    if len(deeper) > 0:

        strongest = deeper.iloc[0]

        print()
        print("DEEPEST COMPARABLE EPISODE")
        print("-" * 110)

        print(
            f"Trough Date       : "
            f"{pd.Timestamp(strongest['Trough_Date']).date()}"
        )

        print(
            f"Maximum Correction: "
            f"{strongest['Correction_%']:.2f}%"
        )

        print(
            f"Signal Reached    : "
            f"{strongest['Peak_to_Trough_Signal']}"
        )

        print(
            f"Peak → Trough     : "
            f"{strongest['Peak_to_Trough_Days']:.0f} days"
        )

        print(
            f"Trough → Recovery : "
            f"{strongest['Trough_to_Recovery_Days']:.0f} days"
            if pd.notna(
                strongest['Trough_to_Recovery_Days']
            )
            else
            "Trough → Recovery : OPEN"
        )

    return diagnostic, opportunities


# =================================================================================================
# OPTIONAL: RUN MULTIPLE FUNDS
# =================================================================================================

def robustness_summary(
    df_nav,
    scheme_codes
):

    rows = []

    for code in scheme_codes:

        try:

            diagnostic, opportunities = robustness_test(
                df_nav,
                code
            )

            rows.append({

                "Scheme_Code":
                    diagnostic["Scheme_Code"],

                "Scheme_Name":
                    diagnostic["Scheme_Name"],

                "Current_Date":
                    diagnostic["Current_Date"],

                "Current_NAV":
                    diagnostic["Current_NAV"],

                "Current_Peak":
                    diagnostic["Current_Peak"],

                "Current_Correction_%":
                    diagnostic["Current_Correction_%"],

                "P50":
                    diagnostic["P50"],

                "P75":
                    diagnostic["P75"],

                "P90":
                    diagnostic["P90"],

                "P95":
                    diagnostic["P95"],

                "Current_Signal":
                    diagnostic["Current_Signal"],

                "Historical_Episodes":
                    diagnostic["Historical_Episode_Count"],

                "Confidence":
                    diagnostic["Confidence"],

                "Deeper_Than_Today":
                    diagnostic["Deeper_Than_Today_Count"],

                "Near_Today":
                    diagnostic["Near_Today_Count"]

            })

        except Exception as e:

            rows.append({

                "Scheme_Code":
                    code,

                "Scheme_Name":
                    "ERROR",

                "Current_Signal":
                    str(e)

            })

    return pd.DataFrame(rows)


# =================================================================================================
# EXAMPLES
# =================================================================================================

# Single fund:
#
# diagnostic_120594, opportunities_120594 = \
#     print_robustness_report(
#         df_nav,
#         120594
#     )


# Multiple funds:
#
test_codes = [
    119063, 141877, 152430, 122639, 147409,
    120594, 120578, 119705, 127042, 138528,
    119218, 119727, 119151, 148623, 120733,
    118834, 135781, 120334, 118825, 153198,
    125354, 120244, 119544, 118989, 130503,
    119714
]

robustness_df = robustness_summary(
    df_nav,
    test_codes
)

display(robustness_df)

,Scheme_Code,Scheme_Name,Current_Date,Current_NAV,Current_Peak,Current_Correction_%,P50,P75,P90,P95,Current_Signal,Historical_Episodes,Confidence,Deeper_Than_Today,Near_Today
0,119063,HDFC Nifty 50 Index Fund - Direct Plan,2026-08-27,235.7810,255.8052,7.827910,5.290830,9.880066,14.962378,17.835379,WATCH,36,HIGH,11,1
1,141877,DSP Nifty 50 Equal Weight Index Fund - Di...,2026-08-27,27.6164,28.4336,2.874064,5.594183,10.180791,16.949078,19.183725,NORMAL,20,HIGH,20,0
2,152430,HDFC NIFTY200 Momentum 30 Index Fund - Di...,2026-08-27,10.5573,12.7984,17.510783,5.028115,7.230951,8.285459,9.033103,STRONG ACCUMULATE,7,LOW,0,0
3,122639,Parag Parikh Flexi Cap Fund - Direct Plan...,2026-08-26,91.0500,95.7732,4.931651,4.914395,6.332936,11.885385,14.668668,WATCH,32,HIGH,16,9
4,147409,Aditya Birla Sun Life Pharma and Healthca...,2026-08-27,40.9600,41.0200,0.146270,4.767102,7.648243,17.220876,20.243386,NORMAL,18,MEDIUM,18,0
5,120594,ICICI Prudential Technology Fund - Direct...,2026-08-26,205.3000,249.9900,17.876715,5.865892,12.581792,16.323011,25.481032,STRONG ACCUMULATE,27,HIGH,2,1
6,120578,SBI TECHNOLOGY OPPORTUNITIES FUND - DIREC...,2026-08-26,235.1978,268.9512,12.550009,5.329085,9.316319,17.420976,23.408808,ACCUMULATE,32,HIGH,6,1
7,119705,SBI COMMA Fund - DIRECT PLAN - Growth,2026-08-27,128.7721,129.5482,0.599082,5.416778,9.422607,22.617981,27.100664,NORMAL,38,HIGH,38,0
8,127042,Motilal Oswal Midcap Fund-Direct Plan-Gro...,2026-08-27,122.7222,130.6636,6.077745,4.756517,7.799307,15.580991,18.866787,WATCH,36,HIGH,14,3
9,138528,PGIM India Global Equity Opportunities Fu...,2026-08-26,56.0600,63.1300,11.199113,5.323958,8.145106,18.516802,26.208018,ACCUMULATE,37,HIGH,7,0


In [19]:
# @title
# ================================================================
# V8.3 — ₹2 LAKH INVESTMENT ALLOCATION ENGINE
# ================================================================

MONTHLY_INVESTMENT_INR = 200000


# ------------------------------------------------
# 1. INVESTMENT SCORE
# ------------------------------------------------
#
# Stronger correction = higher score
# But group overlap reduces the score when several
# funds represent essentially the same exposure.
# ------------------------------------------------

def investment_score(row):

    signal = row.get("Signal", "NORMAL")

    correction = safe_float(
        row.get("Current_Correction_%", np.nan)
    )

    p75 = safe_float(
        row.get("75th_Correction_%", np.nan)
    )

    p90 = safe_float(
        row.get("90th_Correction_%", np.nan)
    )

    p95 = safe_float(
        row.get("95th_Correction_%", np.nan)
    )

    group_pressure = safe_float(
        row.get("Net_Group_Pressure", 0)
    )

    score = 0.0

    # ------------------------------------------------------------
    # SIGNAL
    # ------------------------------------------------------------

    signal_score = {
        "STRONG ACCUMULATE": 100,
        "ACCUMULATE": 70,
        "WATCH": 30,
        "NORMAL": 0,
        "TRIM WATCH": -50,
        "TRIM": -80,
        "STRONG TRIM": -100
    }

    score += signal_score.get(signal, 0)


    # ------------------------------------------------------------
    # CORRECTION DEPTH
    # ------------------------------------------------------------

    if not pd.isna(correction):

        if not pd.isna(p95) and correction >= p95:
            score += 60

        elif not pd.isna(p90) and correction >= p90:
            score += 45

        elif not pd.isna(p75) and correction >= p75:
            score += 30

        elif not pd.isna(p75) and correction >= p75 * 0.75:
            score += 15


    # ------------------------------------------------------------
    # GROUP PRESSURE
    # ------------------------------------------------------------

    if not pd.isna(group_pressure):

        # Positive group pressure supports buying
        score += group_pressure * 10


    return max(score, 0)


# ------------------------------------------------
# 2. CALCULATE RAW SCORES
# ------------------------------------------------

diagnostics_df["Investment_Score"] = (
    diagnostics_df.apply(
        investment_score,
        axis=1
    )
)


# ------------------------------------------------
# 3. ONLY BUY-ELIGIBLE FUNDS
# ------------------------------------------------

buy_mask = diagnostics_df["Signal"].isin(
    [
        "STRONG ACCUMULATE",
        "ACCUMULATE",
        "WATCH"
    ]
)

buy_df = diagnostics_df[
    buy_mask
].copy()


# ------------------------------------------------
# 4. REMOVE ZERO-SCORE FUNDS
# ------------------------------------------------

buy_df = buy_df[
    buy_df["Investment_Score"] > 0
].copy()


# ------------------------------------------------
# 5. GROUP OVERLAP CONTROL
# ------------------------------------------------
#
# Multiple funds inside the same strategic group
# should NOT automatically receive equal allocations.
#
# Example:
#
# HDFC Nifty 50
# ICICI Nifty 50
#
# are effectively one exposure.
#
# Therefore the group gets a budget first.
# ------------------------------------------------

if len(buy_df) > 0:

    group_scores = (
        buy_df
        .groupby("Strategic_Group")[
            "Investment_Score"
        ]
        .sum()
    )

else:

    group_scores = pd.Series(dtype=float)


# ------------------------------------------------
# 6. GROUP ALLOCATION
# ------------------------------------------------

if len(group_scores) > 0:

    total_group_score = group_scores.sum()

    group_budget = (
        group_scores
        / total_group_score
        * MONTHLY_INVESTMENT_INR
    )

else:

    group_budget = pd.Series(
        dtype=float
    )


# ------------------------------------------------
# 7. FUND ALLOCATION WITHIN GROUP
# ------------------------------------------------

buy_df["Group_Budget_INR"] = (
    buy_df["Strategic_Group"]
    .map(group_budget)
    .fillna(0)
)


buy_df["Within_Group_Weight"] = (
    buy_df["Investment_Score"]
    /
    buy_df.groupby(
        "Strategic_Group"
    )["Investment_Score"]
    .transform("sum")
)


buy_df["Investment_INR"] = (
    buy_df["Group_Budget_INR"]
    *
    buy_df["Within_Group_Weight"]
)


# ------------------------------------------------
# 8. ROUND TO ₹1,000
# ------------------------------------------------

buy_df["Investment_INR"] = (
    buy_df["Investment_INR"]
    .div(1000)
    .round()
    .mul(1000)
)


# ------------------------------------------------
# 9. CAP INDIVIDUAL FUND ALLOCATION
# ------------------------------------------------
#
# No single fund gets > 25% of monthly capital.
# This prevents one extreme correction from consuming
# the entire ₹2 lakh.
# ------------------------------------------------

MAX_FUND_ALLOCATION_INR = (
    MONTHLY_INVESTMENT_INR * 0.25
)


buy_df["Investment_INR"] = (
    buy_df["Investment_INR"]
    .clip(
        upper=MAX_FUND_ALLOCATION_INR
    )
)


# ------------------------------------------------
# 10. FINAL NORMALIZATION
# ------------------------------------------------

allocated = buy_df[
    "Investment_INR"
].sum()


if allocated > 0:

    scale = (
        MONTHLY_INVESTMENT_INR
        / allocated
    )

    buy_df["Investment_INR"] = (
        buy_df["Investment_INR"]
        * scale
    )


    buy_df["Investment_INR"] = (
        buy_df["Investment_INR"]
        .div(1000)
        .round()
        .mul(1000)
    )


# ------------------------------------------------
# 11. ADD ₹2 LAKH ALLOCATION BACK TO FULL OUTPUT
# ------------------------------------------------

diagnostics_df["Investment_INR"] = 0


if len(buy_df) > 0:

    diagnostics_df.loc[
        buy_df.index,
        "Investment_INR"
    ] = buy_df["Investment_INR"]


# ------------------------------------------------
# 12. INVESTMENT PERCENTAGE
# ------------------------------------------------

diagnostics_df["Investment_%"] = (
    diagnostics_df["Investment_INR"]
    / MONTHLY_INVESTMENT_INR
    * 100
)


# ------------------------------------------------
# 13. FINAL INVESTMENT ACTION
# ------------------------------------------------

def investment_action(row):

    amount = row["Investment_INR"]

    signal = row["Signal"]

    if amount <= 0:
        return "NO INVESTMENT"

    if signal == "STRONG ACCUMULATE":
        return "INVEST AGGRESSIVELY"

    if signal == "ACCUMULATE":
        return "INVEST"

    if signal == "WATCH":
        return "SMALL INVESTMENT"

    return "NO INVESTMENT"


diagnostics_df[
    "Investment_Action"
] = diagnostics_df.apply(
    investment_action,
    axis=1
)


# ================================================================
# 14. FINAL PORTFOLIO OUTPUT
# ================================================================

portfolio_columns = [

    "Scheme_Code",
    "Scheme_Name",
    "Strategic_Group",

    "Signal",

    "Current_NAV",
    "Current_Correction_%",
    "Correction_Zone",
    "Cycle_Phase",
    "Peak_Recovery_%",

    "Trough_Growth_%",
    "Overshoot_%",

    "Profit_Taking_Signal",
    "Suggested_Trim_%",

    "Group_Signal",
    "Buy_Pressure_%",
    "Trim_Pressure_%",
    "Net_Group_Pressure",

    "Investment_Score",
    "Investment_Action",
    "Investment_INR",
    "Investment_%",

    "Rationale"
]


portfolio_output = diagnostics_df[
    [
        c for c in portfolio_columns
        if c in diagnostics_df.columns
    ]
].copy()


# ================================================================
# 15. DISPLAY ONLY FUNDS RECEIVING MONEY
# ================================================================

investment_output = (
    portfolio_output[
        portfolio_output["Investment_INR"] > 0
    ]
    .sort_values(
        "Investment_INR",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\n"
    + "=" * 150
)

print(
    "V8.3 — ₹2 LAKH MONTHLY INVESTMENT ALLOCATION"
)

print(
    "=" * 150
)

display(
    investment_output
)


print(
    "\n"
    + "=" * 150
)

print(
    f"TOTAL ALLOCATED: "
    f"₹{investment_output['Investment_INR'].sum():,.0f}"
)

print(
    f"UNALLOCATED: "
    f"₹{MONTHLY_INVESTMENT_INR - investment_output['Investment_INR'].sum():,.0f}"
)

print(
    "=" * 150
)


V8.3 — ₹2 LAKH MONTHLY INVESTMENT ALLOCATION


,Scheme_Code,Scheme_Name,Strategic_Group,Signal,Current_NAV,Current_Correction_%,Correction_Zone,Cycle_Phase,Peak_Recovery_%,Trough_Growth_%,Overshoot_%,Profit_Taking_Signal,Suggested_Trim_%,Group_Signal,Buy_Pressure_%,Trim_Pressure_%,Net_Group_Pressure,Investment_Score,Investment_Action,Investment_INR,Investment_%,Rationale
0,152430,HDFC NIFTY200 Momentum 30 Index Fund - Di...,FACTOR_EQUITY,STRONG ACCUMULATE,10.5573,17.510783,ABOVE P95,EXTREME CORRECTION,9.883791,2.383746,-17.510783,NO TRIM,0,GROUP STRONG ACCUMULATE,100.0,0.0,3.00,190.0,INVEST AGGRESSIVELY,17000,8.5,Correction 17.51% | Median 1.39% | P75 2....
1,138528,PGIM India Global Equity Opportunities Fu...,INTERNATIONAL,STRONG ACCUMULATE,56.0600,11.199113,ABOVE P95,EXTREME CORRECTION,2.884615,0.376007,-11.199113,NO TRIM,0,GROUP STRONG ACCUMULATE,100.0,0.0,3.00,190.0,INVEST AGGRESSIVELY,17000,8.5,Correction 11.20% | Median 1.05% | P75 2....
2,120594,ICICI Prudential Technology Fund - Direct...,SECTOR_EQUITY,STRONG ACCUMULATE,205.3000,17.876715,ABOVE P95,EXTREME CORRECTION,-1.753188,-0.373659,-17.876715,NO TRIM,0,GROUP ACCUMULATE,71.4,0.0,1.86,178.6,INVEST AGGRESSIVELY,16000,8.0,Correction 17.88% | Median 0.93% | P75 2....
3,120578,SBI TECHNOLOGY OPPORTUNITIES FUND - DIREC...,SECTOR_EQUITY,STRONG ACCUMULATE,235.1978,12.550009,ABOVE P95,EXTREME CORRECTION,-4.314314,-0.590041,-12.550009,NO TRIM,0,GROUP ACCUMULATE,71.4,0.0,1.86,178.6,INVEST AGGRESSIVELY,16000,8.0,Correction 12.55% | Median 0.97% | P75 2....
4,119063,HDFC Nifty 50 Index Fund - Direct Plan,CORE_EQUITY,STRONG ACCUMULATE,235.7810,7.827910,ABOVE P95,EXTREME CORRECTION,-6.727997,-0.532519,-7.827910,NO TRIM,0,GROUP ACCUMULATE,75.0,0.0,1.75,177.5,INVEST AGGRESSIVELY,16000,8.0,Correction 7.83% | Median 0.91% | P75 1.8...
5,120244,ICICI Prudential Banking and Financial Se...,SECTOR_EQUITY,STRONG ACCUMULATE,149.6900,5.944078,ABOVE P95,EXTREME CORRECTION,10.922787,0.780987,-5.944078,NO TRIM,0,GROUP ACCUMULATE,71.4,0.0,1.86,178.6,INVEST AGGRESSIVELY,16000,8.0,Correction 5.94% | Median 1.01% | P75 2.2...
6,127042,Motilal Oswal Midcap Fund-Direct Plan-Gro...,GROWTH_EQUITY,STRONG ACCUMULATE,122.7222,6.077745,ABOVE P95,EXTREME CORRECTION,1.491019,0.098041,-6.077745,NO TRIM,0,GROUP ACCUMULATE,60.0,0.0,1.60,176.0,INVEST AGGRESSIVELY,15000,7.5,Correction 6.08% | Median 0.96% | P75 2.0...
7,122639,Parag Parikh Flexi Cap Fund - Direct Plan...,GROWTH_EQUITY,STRONG ACCUMULATE,91.0500,4.931651,ABOVE P95,EXTREME CORRECTION,12.051244,0.715907,-4.931651,NO TRIM,0,GROUP ACCUMULATE,60.0,0.0,1.60,176.0,INVEST AGGRESSIVELY,15000,7.5,Correction 4.93% | Median 0.62% | P75 1.3...
8,118825,Mirae Asset Large Cap Fund - Direct Plan ...,CORE_EQUITY,ACCUMULATE,128.5680,4.220987,P90 → P95,EXTREME CORRECTION,-10.837246,-0.429052,-4.220987,NO TRIM,0,GROUP ACCUMULATE,75.0,0.0,1.75,132.5,INVEST,12000,6.0,Correction 4.22% | Median 0.86% | P75 1.7...
9,119218,DSP Large & Mid Cap Fund - Direct Plan - ...,CORE_EQUITY,ACCUMULATE,705.6970,3.389659,P75 → P90,DEEP CORRECTION,-1.964337,-0.067547,-3.389659,NO TRIM,0,GROUP ACCUMULATE,75.0,0.0,1.75,117.5,INVEST,10000,5.0,Correction 3.39% | Median 0.89% | P75 1.9...



TOTAL ALLOCATED: ₹200,000
UNALLOCATED: ₹0


In [6]:
# @title
# ================================================================
# HISTORICAL BUYING-OPPORTUNITY ROBUSTNESS TEST
# V8.x diagnostic
#
# Purpose:
#   For selected STRONG ACCUMULATE funds:
#   1. Find today's correction
#   2. Find historical dates with equal/deeper corrections
#   3. Calculate the signal using ONLY information available
#      before each historical date
#   4. Compare historical opportunities with today's opportunity
#
# IMPORTANT:
#   This avoids look-ahead bias.
# ================================================================

import pandas as pd
import numpy as np


# ---------------------------------------------------------------
# FUNDS TO TEST
# ---------------------------------------------------------------

TEST_FUNDS = {
    120594: "ICICI Prudential Technology Fund",
    152430: "HDFC NIFTY200 Momentum 30 Index Fund",
    122639:	"Parag Parikh Flexi Cap Fund - Direct Plan",
}


# ---------------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------------

MIN_EPISODE_CORRECTION = 2.0
MIN_EPISODE_DAYS = 3

# Historical signal thresholds
P75_THRESHOLD = 75
P90_THRESHOLD = 90
P95_THRESHOLD = 95

# How close to today's correction counts as "comparable"
COMPARABLE_TOLERANCE = 1.0

# Minimum separation between historical opportunities.
# Prevents displaying every day of the same correction episode.
MIN_OPPORTUNITY_GAP_DAYS = 20


# ---------------------------------------------------------------
# PREPARE NAV DATA
# ---------------------------------------------------------------

nav = df_nav.copy()

nav["Date"] = pd.to_datetime(nav["Date"])
nav["Scheme_Code"] = nav["Scheme_Code"].astype(int)
nav["NAV"] = pd.to_numeric(nav["NAV"], errors="coerce")

nav = (
    nav.dropna(subset=["Date", "Scheme_Code", "NAV"])
       .sort_values(["Scheme_Code", "Date"])
       .reset_index(drop=True)
)


# ---------------------------------------------------------------
# SIGNAL FUNCTION
# ---------------------------------------------------------------

def historical_signal(correction, p50, p75, p90, p95):

    if pd.isna(correction):
        return "NORMAL"

    if correction >= p95:
        return "STRONG ACCUMULATE"

    elif correction >= p90:
        return "ACCUMULATE"

    elif correction >= p75:
        return "WATCH"

    elif correction >= p50:
        return "NORMAL"

    else:
        return "NORMAL"


# ---------------------------------------------------------------
# BUILD EPISODES
#
# A correction episode starts after a meaningful decline from
# a running peak and closes when the previous peak is recovered.
# ---------------------------------------------------------------

def build_historical_episodes(g):

    g = g.sort_values("Date").reset_index(drop=True)

    peak_nav = -np.inf
    peak_date = None

    episodes = []

    active = False
    trough_nav = None
    trough_date = None

    for _, r in g.iterrows():

        date = r["Date"]
        value = r["NAV"]

        # establish / update running peak
        if value >= peak_nav:

            # if an episode was active, this is recovery
            if active and peak_nav > 0:

                recovery_days = (date - trough_date).days
                peak_to_trough_days = (trough_date - peak_date).days
                total_days = (date - peak_date).days

                correction = (
                    (peak_nav - trough_nav) /
                    peak_nav * 100
                )

                if (
                    correction >= MIN_EPISODE_CORRECTION
                    and peak_to_trough_days >= MIN_EPISODE_DAYS
                ):

                    episodes.append({
                        "Peak_Date": peak_date,
                        "Peak_NAV": peak_nav,
                        "Trough_Date": trough_date,
                        "Trough_NAV": trough_nav,
                        "Recovery_Date": date,
                        "Correction_%": correction,
                        "Peak_to_Trough_Days": peak_to_trough_days,
                        "Trough_to_Recovery_Days": recovery_days,
                        "Total_Recovery_Days": total_days
                    })

                active = False
                trough_nav = None
                trough_date = None

            peak_nav = value
            peak_date = date

        else:

            correction = (
                (peak_nav - value) /
                peak_nav * 100
            )

            # start meaningful correction
            if not active and correction >= MIN_EPISODE_CORRECTION:
                active = True
                trough_nav = value
                trough_date = date

            # update trough
            elif active and value < trough_nav:
                trough_nav = value
                trough_date = date

    return pd.DataFrame(episodes)


# ---------------------------------------------------------------
# HISTORICAL SIGNAL SNAPSHOT
#
# IMPORTANT:
# Percentiles use ONLY episodes that were completed BEFORE
# the historical date.
# ---------------------------------------------------------------

def build_opportunity_history(g):

    g = g.sort_values("Date").reset_index(drop=True)

    episodes = build_historical_episodes(g)

    if episodes.empty:
        return pd.DataFrame()

    results = []

    running_peak = -np.inf
    running_peak_date = None

    for _, r in g.iterrows():

        date = r["Date"]
        nav_value = r["NAV"]

        if nav_value >= running_peak:
            running_peak = nav_value
            running_peak_date = date

        if running_peak <= 0:
            continue

        correction = (
            (running_peak - nav_value) /
            running_peak * 100
        )

        # only episodes COMPLETED before this date
        prior = episodes[
            episodes["Recovery_Date"] < date
        ]

        if len(prior) < 3:
            continue

        corrections = prior["Correction_%"]

        p50 = corrections.quantile(0.50)
        p75 = corrections.quantile(0.75)
        p90 = corrections.quantile(0.90)
        p95 = corrections.quantile(0.95)

        signal = historical_signal(
            correction,
            p50,
            p75,
            p90,
            p95
        )

        results.append({
            "Date": date,
            "NAV": nav_value,
            "Peak_NAV": running_peak,
            "Peak_Date": running_peak_date,
            "Correction_%": correction,
            "P50": p50,
            "P75": p75,
            "P90": p90,
            "P95": p95,
            "Signal": signal,
            "Historical_Episodes_Available": len(prior)
        })

    return pd.DataFrame(results)


# ---------------------------------------------------------------
# FIND DISTINCT HISTORICAL BUYING OPPORTUNITIES
#
# We keep the deepest observation within each correction episode
# rather than reporting 20 consecutive days.
# ---------------------------------------------------------------

def find_opportunities(history, current_correction):

    if history.empty:
        return pd.DataFrame()

    h = history.copy()

    # equal or deeper than current opportunity
    h["Comparable"] = (
        h["Correction_%"] >=
        current_correction - COMPARABLE_TOLERANCE
    )

    h = h[h["Comparable"]].copy()

    if h.empty:
        return h

    h = h.sort_values("Date")

    selected = []
    last_date = None

    for _, r in h.iterrows():

        if (
            last_date is None
            or (r["Date"] - last_date).days >= MIN_OPPORTUNITY_GAP_DAYS
        ):
            selected.append(r)
            last_date = r["Date"]

        else:

            # replace previous observation if this one is deeper
            if r["Correction_%"] > selected[-1]["Correction_%"]:
                selected[-1] = r
                last_date = r["Date"]

    return pd.DataFrame(selected)


# ---------------------------------------------------------------
# RUN TEST
# ---------------------------------------------------------------

all_results = []
all_opportunities = []

for code, fund_name in TEST_FUNDS.items():

    g = nav[nav["Scheme_Code"] == code].copy()

    if g.empty:
        print(f"\nNOT FOUND: {code} - {fund_name}")
        continue

    history = build_opportunity_history(g)

    if history.empty:
        print(f"\nINSUFFICIENT HISTORY: {code} - {fund_name}")
        continue

    latest = g.iloc[-1]

    current_nav = latest["NAV"]
    current_date = latest["Date"]

    current_peak = g.loc[g["NAV"].cummax().idxmax(), "NAV"]

    current_correction = (
        (current_peak - current_nav) /
        current_peak * 100
    )

    # Current historical percentile using all COMPLETED episodes
    episodes = build_historical_episodes(g)

    p50 = episodes["Correction_%"].quantile(0.50)
    p75 = episodes["Correction_%"].quantile(0.75)
    p90 = episodes["Correction_%"].quantile(0.90)
    p95 = episodes["Correction_%"].quantile(0.95)

    current_signal = historical_signal(
        current_correction,
        p50,
        p75,
        p90,
        p95
    )

    opportunities = find_opportunities(
        history,
        current_correction
    )

    print("\n" + "=" * 100)
    print(f"BUYING OPPORTUNITY ROBUSTNESS TEST")
    print(f"{code} | {fund_name}")
    print("=" * 100)

    print(f"Current date       : {current_date.date()}")
    print(f"Current NAV        : {current_nav:.4f}")
    print(f"Current peak       : {current_peak:.4f}")
    print(f"Current correction : {current_correction:.2f}%")
    print(f"Current P50        : {p50:.2f}%")
    print(f"Current P75        : {p75:.2f}%")
    print(f"Current P90        : {p90:.2f}%")
    print(f"Current P95        : {p95:.2f}%")
    print(f"CURRENT SIGNAL     : {current_signal}")

    print("\nHistorical opportunities equal to or better than today")
    print("-" * 100)

    if opportunities.empty:

        print("No comparable historical opportunities found.")

    else:

        out = opportunities[
            [
                "Date",
                "NAV",
                "Peak_NAV",
                "Correction_%",
                "P75",
                "P90",
                "P95",
                "Signal",
                "Historical_Episodes_Available"
            ]
        ].copy()

        out["Date"] = out["Date"].dt.strftime("%Y-%m-%d")

        out = out.sort_values(
            "Correction_%",
            ascending=False
        )

        print(out.to_string(index=False))

        out["Scheme_Code"] = code
        out["Scheme_Name"] = fund_name

        all_opportunities.append(out)

    # current snapshot
    all_results.append({
        "Scheme_Code": code,
        "Scheme_Name": fund_name,
        "Current_Date": current_date,
        "Current_NAV": current_nav,
        "Current_Peak": current_peak,
        "Current_Correction_%": current_correction,
        "P50": p50,
        "P75": p75,
        "P90": p90,
        "P95": p95,
        "Current_Signal": current_signal,
        "Comparable_Historical_Opportunities":
            len(opportunities)
    })


# ---------------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------------

robustness_summary = pd.DataFrame(all_results)

if all_opportunities:
    historical_buying_opportunities = pd.concat(
        all_opportunities,
        ignore_index=True
    )
else:
    historical_buying_opportunities = pd.DataFrame()


print("\n\n")
print("=" * 100)
print("ROBUSTNESS SUMMARY")
print("=" * 100)

print(
    robustness_summary.to_string(index=False)
)

print("\n")
print("=" * 100)
print("INTERPRETATION")
print("=" * 100)

print("""
The test asks:

1. How deep is today's correction?
2. Has this fund historically reached the same or deeper correction?
3. What signal would the engine have produced at those historical dates?
4. How many distinct historical opportunities were there?
5. Were today's conditions genuinely exceptional or merely normal?

IMPORTANT:
Historical percentile thresholds are calculated only from correction
episodes completed BEFORE each historical observation.

Therefore a historical STRONG ACCUMULATE is a signal the engine could
actually have known at that time, rather than a signal contaminated
by future data.
""")


BUYING OPPORTUNITY ROBUSTNESS TEST
120594 | ICICI Prudential Technology Fund
Current date       : 2026-08-26
Current NAV        : 205.3000
Current peak       : 249.9900
Current correction : 17.88%
Current P50        : 5.23%
Current P75        : 7.68%
Current P90        : 15.78%
Current P95        : 21.21%
CURRENT SIGNAL     : ACCUMULATE

Historical opportunities equal to or better than today
----------------------------------------------------------------------------------------------------
      Date    NAV  Peak_NAV  Correction_%       P75       P90       P95            Signal  Historical_Episodes_Available
2020-03-23  42.75     65.86     35.089584 11.738790 15.606617 16.064311 STRONG ACCUMULATE                             18
2022-06-17 134.11    189.26     29.139808  6.837983 15.563406 16.529970 STRONG ACCUMULATE                             29
2022-07-15 134.83    189.26     28.759379  6.837983 15.563406 16.529970 STRONG ACCUMULATE                             29
2023-04-19 136.03  